In [17]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from deepface import DeepFace

# Paths
DATASET_PATH = r"C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset"
OUTPUT_PATH  = r"C:\Users\hp\StressDetection\valence_features55-56.csv"

# Video settings
FPS            = 35
WINDOW_SIZE    = 140  # 4 seconds
STEP_SIZE      = 35   # 1 second
DEEPFACE_EVERY = 17   # run DeepFace every 17 frames (~0.5 seconds)

print("Setup complete!")

Setup complete!


In [18]:
def compute_valence(emotion_dict):
    """
    Convert DeepFace 7 emotions to valence scale
    Based on Russell's Circumplex Model of Affect (1980)
    
    Positive emotions: happy + (surprise × 0.5)
    Negative emotions: angry + disgust + fear + sad
    Neutral: neutral
    
    Formula: valence = (positive - negative) / total
    Range: -1.0 (very stressed) to +1.0 (very relaxed)
    
    Note: surprise × 0.5 because surprise is ambiguous
    (can be positive or negative)
    """
    happy    = emotion_dict.get('happy', 0)
    surprise = emotion_dict.get('surprise', 0)
    angry    = emotion_dict.get('angry', 0)
    disgust  = emotion_dict.get('disgust', 0)
    fear     = emotion_dict.get('fear', 0)
    sad      = emotion_dict.get('sad', 0)
    neutral  = emotion_dict.get('neutral', 0)
    
    positive = happy + (surprise * 0.5)
    negative = angry + disgust + fear + sad
    total    = positive + negative + neutral
    
    if total == 0:
        return 0.0
    
    return (positive - negative) / total

print("Valence function defined!")

Valence function defined!


In [19]:
def apply_sliding_window_valence(valence_list, total_frames,
                                  window_size=140, step_size=35):
    """
    Apply sliding window to valence estimates
    
    DeepFace runs every 17 frames → ~8 estimates per 4-second window
    
    For each window compute 4 statistics:
    mean, std, min, max
    
    Returns list of window dictionaries
    """
    windows   = []
    start     = 0
    window_id = 0
    
    while start + window_size <= total_frames:
        end = start + window_size
        
        window_valences = [v for (f, v) in valence_list if start <= f < end]
        
        if len(window_valences) == 0:
            window_valences = [0.0]
        
        row = {
            'window_id':    window_id,
            'valence_mean': np.mean(window_valences),
            'valence_std':  np.std(window_valences),
            'valence_min':  np.min(window_valences),
            'valence_max':  np.max(window_valences)
        }
        
        windows.append(row)
        start     += step_size
        window_id += 1
    
    return windows

print("Sliding window valence function defined!")

Sliding window valence function defined!


In [20]:
def process_video_valence(video_path):
    """
    Process one video and extract valence estimates
    
    DeepFace runs every 17 frames (~0.5 seconds)
    If DeepFace fails on a frame → use last known valence
    
    Returns list of window rows
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"ERROR: Cannot open {video_path}")
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    valence_list = []
    last_valence = 0.0
    
    for frame_idx in tqdm(range(total_frames), desc="Processing frames"):
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % DEEPFACE_EVERY == 0:
            try:
                result       = DeepFace.analyze(
                    frame,
                    actions=['emotion'],
                    enforce_detection=False,
                    silent=True
                )
                emotion_dict = result[0]['emotion']
                valence      = compute_valence(emotion_dict)
                last_valence = valence
            except Exception:
                valence = last_valence
            
            valence_list.append((frame_idx, valence))
    
    cap.release()
    
    windows = apply_sliding_window_valence(valence_list, total_frames)
    return windows

print("Video valence processing function defined!")

Video valence processing function defined!


In [21]:
def process_participant_valence(participant_id):
    """
    Process T1, T2, T3 valence for one participant
    
    Tasks:
    T1 = rest    → label 0
    T2 = speech  → label 1
    T3 = arithmetic → label 1
    
    Returns list of all window rows
    """
    all_rows = []
    tasks    = {'T1': 0, 'T2': 1, 'T3': 1}
    
    for task, label in tasks.items():
        video_name = f"vid_s{participant_id}_{task}.avi"
        video_path = os.path.join(
            DATASET_PATH,
            f"Participant{participant_id}",
            video_name
        )
        
        if not os.path.exists(video_path):
            print(f"WARNING: {video_path} not found — skipping")
            continue
        
        print(f"\nProcessing Participant {participant_id} - {task}...")
        windows = process_video_valence(video_path)
        
        for row in windows:
            row['participant'] = participant_id
            row['task']        = task
            row['label']       = label
        
        all_rows.extend(windows)
    
    return all_rows

print("Participant valence function defined!")

Participant valence function defined!


In [22]:
def apply_baseline_normalization_valence(df):
    """
    Per-subject baseline normalization for valence
    
    Same logic as geometric normalization:
    T1 = raw values (baseline reference)
    T2 = deviation from personal valence baseline
    T3 = deviation from personal valence baseline
    """
    feature_cols  = ['valence_mean', 'valence_std', 'valence_min', 'valence_max']
    normalized_df = df.copy()
    normalized_df[feature_cols] = normalized_df[feature_cols].astype(float)
    
    for participant_id in df['participant'].unique():
        t1_mask     = (normalized_df['participant'] == participant_id) & \
                      (normalized_df['task'] == 'T1')
        t1_baseline = normalized_df[t1_mask][feature_cols].mean()
        
        for task in ['T2', 'T3']:
            task_mask = (normalized_df['participant'] == participant_id) & \
                        (normalized_df['task'] == task)
            normalized_df.loc[task_mask, feature_cols] = \
                normalized_df.loc[task_mask, feature_cols] - t1_baseline
    
    return normalized_df

print("Valence normalization function defined!")

Valence normalization function defined!


In [23]:
def run_valence_pipeline(participant_ids):
    """
    Full valence pipeline — processes participants and saves to CSV
    
    Same strategy as geometric pipeline:
    - Check if CSV exists → append (never overwrite)
    - Save after each participant → safe to delete videos after
    
    Usage:
    run_valence_pipeline([1, 2, 3])  → process participants 1, 2, 3
    run_valence_pipeline([4])        → process participant 4 only
    """
    for participant_id in participant_ids:
        print(f"\n{'='*50}")
        print(f"PARTICIPANT {participant_id}")
        print(f"{'='*50}")
        
        rows = process_participant_valence(participant_id)
        
        if len(rows) == 0:
            print(f"WARNING: No data for participant {participant_id} — skipping")
            continue
        
        df_participant = pd.DataFrame(rows)
        df_participant = apply_baseline_normalization_valence(df_participant)
        
        cols = ['participant', 'task', 'window_id', 'label'] + \
               ['valence_mean', 'valence_std', 'valence_min', 'valence_max']
        df_participant = df_participant[cols]
        
        if os.path.exists(OUTPUT_PATH):
            df_participant.to_csv(OUTPUT_PATH, index=False, mode='a', header=False)
        else:
            df_participant.to_csv(OUTPUT_PATH, index=False, mode='w')
        
        print(f"Participant {participant_id} saved — {len(df_participant)} rows")
        print(f"CSV updated at: {OUTPUT_PATH}")
    
    print(f"\n{'='*50}")
    print("ALL PARTICIPANTS DONE!")
    print(f"CSV saved to: {OUTPUT_PATH}")

print("Full valence pipeline defined!")

Full valence pipeline defined!


In [24]:
run_valence_pipeline([55, 56])


PARTICIPANT 55

Processing Participant 55 - T1...


Processing frames: 100%|██████████████████████████████████████████████████████████▉| 6325/6326 [04:25<00:00, 23.84it/s]



Processing Participant 55 - T2...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [06:20<00:00, 16.62it/s]



Processing Participant 55 - T3...


...


Processing Participant 56 - T2...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [04:28<00:00, 23.52it/s]



Processing Participant 56 - T3...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [03:46<00:00, 27.97it/s]

Participant 56 saved — 531 rows
CSV updated at: C:\Users\hp\StressDetection\valence_features55-56.csv

ALL PARTICIPANTS DONE!
CSV saved to: C:\Users\hp\StressDetection\valence_features55-56.csv


In [18]:
import pandas as pd

geo = pd.read_csv(r"C:\Users\hp\StressDetection\geometric_features37-38.csv")
val = pd.read_csv(r"C:\Users\hp\StressDetection\valence_features37-38.csv")

print("Geometric:")
print(geo['participant'].value_counts().sort_index())
print(f"Total rows: {len(geo)}")

print("\nValence:")
print(val['participant'].value_counts().sort_index())
print(f"Total rows: {len(val)}")

Geometric:
participant
37    531
38    531
Name: count, dtype: int64
Total rows: 1062

Valence:
participant
37    531
38    531
Name: count, dtype: int64
Total rows: 1062


In [19]:
import pandas as pd

# Paths
GEO_NEW    = r"C:\Users\hp\StressDetection\geometric_features11,12,13.csv"
VAL_NEW    = r"C:\Users\hp\StressDetection\valence_features11,12,13.csv"
GEO_ALL    = r"C:\Users\hp\StressDetection\geometric_features.csv"
VAL_ALL    = r"C:\Users\hp\StressDetection\valence_features.csv"
MASTER     = r"C:\Users\hp\StressDetection\master_features.csv"

# Load new participants
df_geo_new = pd.read_csv(GEO_NEW)
df_val_new = pd.read_csv(VAL_NEW)

# Merge new participants into one
df_new = pd.merge(
    df_geo_new,
    df_val_new[['participant', 'task', 'window_id',
                'valence_mean', 'valence_std', 'valence_min', 'valence_max']],
    on=['participant', 'task', 'window_id']
)

# Append to master
df_master = pd.read_csv(MASTER)
df_master = pd.concat([df_master, df_new], ignore_index=True)
df_master.to_csv(MASTER, index=False)

# Append to geometric_features.csv
df_geo_new.to_csv(GEO_ALL, index=False, mode='a', header=False)

# Append to valence_features.csv
df_val_new.to_csv(VAL_ALL, index=False, mode='a', header=False)

print(f"Master CSV total rows: {len(df_master)}")
print(f"\nParticipants in master:")
print(df_master['participant'].value_counts().sort_index())

Master CSV total rows: 6903

Participants in master:
participant
1     531
2     531
3     531
4     531
5     531
6     531
7     531
8     531
9     531
10    531
11    531
12    531
13    531
Name: count, dtype: int64
